# 06. 80 Expert 모델: 수학 1문제와 LCB 2문제

기존에 비교한 MATH-500 11번 다항식 문제(`math500-test-11`, 정답 `3/56`)와 LCB `3674`, `3677`을 시험합니다. 세 문제는 모델을 한 번만 적재해 생성하고, 모델 종료 후 채점합니다. 이는 전체 벤치마크 점수가 아닙니다.

이전 수학 실행은 context 16,384 / 최대 생성 8,192였고, 이 실행은 context 요청 65,535 / 최대 생성 32,768입니다. 문제는 같지만 속도와 점수를 동일 조건으로 비교할 수 없습니다. 세 문제라도 시간이 오래 걸릴 수 있습니다.

## Colab 환경 자동 준비
소스 ZIP과 데이터 ZIP을 Google Drive의 `MyDrive/Putare` 폴더에 업로드하세요.
설정 셀에서 Drive를 마운트하고 환경 준비 셀을 실행하세요. 모델은 `MODEL_PATH`에 별도로 준비합니다.
준비 셀은 단독 재실행할 수 있습니다. 이전 분할 업로드 재조립 셀은 자동 실행하지 않습니다.


In [23]:
# 2026-11-09: 설치와 빌드는 명시적으로 켰을 때만 실행한다.
SETUP_COLAB = True

from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = '/content/drive/MyDrive/models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf'

if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
SOURCE_ZIP = '/content/drive/MyDrive/Putare/putare-source-7a9774217647.zip'
DATA_ZIP = '/content/drive/MyDrive/Putare/putare-data-0cce743764ad.zip'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
from pathlib import Path

current = Path.cwd()
print("현재 경로:", current)

for folder in (current, current.parent):
    print(f"\n[{folder}]")
    for item in sorted(folder.iterdir()):
        print(f"{'📁' if item.is_dir() else '📄'} {item.name}")

현재 경로: /content/Putare/0812c0c257e2-0cce743764ad

[/content/Putare/0812c0c257e2-0cce743764ad]
📁 .cache
📁 download
📁 moe_prune
📄 requirements-research.txt
📁 runs
📁 tools

[/content/Putare]
📁 0812c0c257e2-0cce743764ad


In [25]:
RUN_REASSEMBLY = False
if RUN_REASSEMBLY:
    from pathlib import Path
    import json, hashlib

    parts = Path('/content/e80-upload-parts')
    info = json.loads((parts / 'transfer.json').read_text())
    target = Path('/content/Putare/models') / info['filename']
    temporary = target.with_suffix('.uploading')

    for item in info['parts']:
        assert (parts / item['name']).stat().st_size == item['bytes'], item['name']
    assert not target.exists(), '완성 모델이 이미 있습니다'
    target.parent.mkdir(parents=True, exist_ok=True)

    digest = hashlib.sha256()
    with temporary.open('xb') as output:
        for item in info['parts']:
            with (parts / item['name']).open('rb') as source:
                while block := source.read(8 * 1024**2):
                    output.write(block)
                    digest.update(block)

    assert digest.hexdigest() == info['sha256'], '전송 파일 검증 실패'
    temporary.rename(target)
    print('모델 준비 완료:', target)

In [26]:
import hashlib, json, os, shutil, sys, zipfile
from pathlib import Path, PurePosixPath

def sha256_file(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def restore_archives(root, archives):
    """모든 ZIP과 기존 경로를 먼저 검증하고 누락된 파일만 복원한다."""
    # 2026-11-09: 불완전한 전송과 경로 탈출·기존 데이터 덮어쓰기를 함께 방지한다.
    root = Path(root)
    if root.is_symlink():
        raise ValueError('복원 위치가 심볼릭 링크입니다')
    validated = []
    used = set()
    try:
        for archive_path, expected_hash in archives:
            archive_path = Path(archive_path)
            if not archive_path.is_file():
                raise FileNotFoundError(f'Drive에 ZIP을 업로드하세요: {archive_path}')
            if sha256_file(archive_path) != expected_hash:
                raise ValueError(f'ZIP SHA256 불일치: {archive_path}')
            archive = zipfile.ZipFile(archive_path)
            validated.append((archive, {}))
            names = archive.namelist()
            if len(names) != len(set(names)) or '_putare_inventory.json' not in names:
                raise ValueError('ZIP 파일 목록에 중복 또는 인벤토리 누락')
            inventory = json.loads(archive.read('_putare_inventory.json'))
            if not isinstance(inventory, dict) or set(names) != set(inventory) | {'_putare_inventory.json'}:
                raise ValueError('ZIP 인벤토리 불일치')
            for name, digest in inventory.items():
                path = PurePosixPath(name)
                if (not isinstance(name, str) or path.is_absolute() or not path.parts
                        or any(p in ('', '.', '..') for p in name.split('/')) or '\\' in name or ':' in name):
                    raise ValueError('안전하지 않은 ZIP 경로')
                info = archive.getinfo(name)
                if info.is_dir() or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('ZIP에 링크 또는 디렉터리가 있습니다')
                if name in used:
                    raise ValueError('소스와 데이터 ZIP에 중복 경로가 있습니다: ' + name)
                used.add(name)
                with archive.open(info) as stream:
                    if hashlib.file_digest(stream, 'sha256').hexdigest() != digest:
                        raise ValueError('ZIP 파일 SHA256 불일치: ' + name)
                target = root.joinpath(*path.parts)
                for part in [root, *[root.joinpath(*path.parts[:i]) for i in range(1, len(path.parts) + 1)]]:
                    if part.is_symlink():
                        raise ValueError('복원 경로에 심볼릭 링크가 있습니다: ' + str(part))
                    if part != target and part.exists() and not part.is_dir():
                        raise FileExistsError('복원 폴더 위치가 파일입니다: ' + str(part))
                if target.exists() and (not target.is_file() or sha256_file(target) != digest):
                    raise FileExistsError('기존 파일이 달라 덮어쓰지 않습니다: ' + str(target))
                validated[-1][1][name] = target
        for archive, paths in validated:
            for name, target in paths.items():
                if not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(name) as stream, target.open('xb') as output:
                        shutil.copyfileobj(stream, output)
    except (zipfile.BadZipFile, KeyError, UnicodeError) as error:
        raise ValueError('ZIP이 손상되었습니다') from error
    finally:
        for archive, _ in validated:
            archive.close()
    return len(used)


if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
    _source_sha256 = '7a97742176470de8c5f01a8d81ea391e945360fd1c007cba19834aab6f879bbc'
    _data_sha256 = '0cce743764ad01a7c8e9fa3016e02b51ca85959fd57fda383e5528faa4c712d1'
    ROOT = Path('/content/Putare') / (_source_sha256[:12] + '-' + _data_sha256[:12])
    if not (3, 12) <= sys.version_info[:2] < (3, 13):
        raise RuntimeError('Python 3.12가 필요합니다. Colab 런타임 버전을 확인하세요.')
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Google Drive가 마운트되지 않았습니다. 설정 셀에서 직접 마운트하세요.')
    # 2026-11-09: 같은 커널의 이전 버전 모듈 혼용을 막는다.
    for _name, _module in list(sys.modules.items()):
        if _name not in ('tools', 'moe_prune') and not _name.startswith(('tools.', 'moe_prune.')):
            continue
        _file = getattr(_module, '__file__', None)
        if _file and not Path(_file).resolve().is_relative_to(ROOT.resolve()):
            raise RuntimeError('다른 Putare 버전이 이미 로드되었습니다. 커널을 다시 시작하세요: ' + _name)
        _paths = getattr(_module, '__path__', ()) if not _file else ()
        if any(not Path(p).resolve().is_relative_to(ROOT.resolve()) for p in _paths):
            raise RuntimeError('다른 Putare 버전이 이미 로드되었습니다. 커널을 다시 시작하세요: ' + _name)
    restore_archives(ROOT, [(SOURCE_ZIP, _source_sha256), (DATA_ZIP, _data_sha256)])
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from tools.colab_setup import prepare_environment
    _PUTARE_SETUP_STATE = prepare_environment(ROOT, globals().get('_PUTARE_SETUP_STATE'))
    _expected_model = Path(MODEL_PATH) if MODEL_PATH else ROOT / 'models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf'
    print('환경 준비 완료. 모델 경로:', _expected_model, '| 존재:', _expected_model.is_file())
    if not _expected_model.is_file():
        print('추론 전 모델 파일을 위 경로에 준비하세요.')
    _PUTARE_SETUP_READY = True


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, Compute Capability: 120


CalledProcessError: Command '['/content/Putare/0812c0c257e2-0cce743764ad/.cache/putare-lcb/venv/bin/python', '-I', '-m', 'pip', '--isolated', 'install', '--disable-pip-version-check', '--no-input', 'numpy>=2,<3', 'tqdm>=4,<5', 'datasets>=3.2,<4']' returned non-zero exit status 1.

In [30]:
from pathlib import Path
import sys
import subprocess

ROOT = Path.cwd().resolve()
if ROOT.name == 'ipynb': ROOT = ROOT.parent
assert (ROOT / 'moe_prune').is_dir(), 'Repository root required'
sys.path.insert(0, str(ROOT))
from tools import benchmark80 as trial
from tools import manual80

INSTALL_DEPENDENCIES = False
PREPARE_GRADER = False
PREPARE_ENGINE = False
BUILD_ENGINE = False
RUN_GENERATION = True
RUN_GRADING = True
RUN_SHOW_TEXT = True
MODEL = Path(MODEL_PATH) if MODEL_PATH else manual80.MODEL
ENGINE = manual80.ENGINE
RUN_DIR = trial.DIRECTORY
GRADER_VENV = ROOT / '.cache/putare-lcb/venv'

print({'model': str(MODEL), 'model_present': MODEL.is_file(),
       'engine': str(ENGINE), 'engine_present': ENGINE.is_file(),
       'run_dir': str(RUN_DIR), 'run_exists': RUN_DIR.exists(), 'case_ids': trial.IDS})


{'model': '/content/drive/MyDrive/models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf', 'model_present': True, 'engine': '/content/Putare/0812c0c257e2-0cce743764ad/.cache/llama-build/bin/llama-expert-research', 'engine_present': True, 'run_dir': '/content/Putare/0812c0c257e2-0cce743764ad/runs/gemma4-ordered-e80-trial3-colab-a100-001', 'run_exists': False, 'case_ids': ['math500-test-11', 'livecodebench:3674', 'livecodebench:3677']}


## 환경 준비

Colab A100 커널과 저장소·80 Expert GGUF·05번 manifest·기존 LCB 묶음이 필요합니다. 필요한 플래그를 켜 Python 의존성, CUDA 엔진, 공식 채점기를 준비합니다. 기존 PyTorch는 교체하지 않습니다.

In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', '--isolated', 'install', '-r', str(ROOT / 'requirements-research.txt')], check=True)
if PREPARE_ENGINE:
    command = [sys.executable, str(ROOT / 'tools/native/prepare.py'), '--backend', 'cuda']
    if BUILD_ENGINE: command.append('--execute')
    subprocess.run(command, cwd=ROOT, check=True)
if PREPARE_GRADER:
    from moe_prune.benchmark_data import prepare_grader
    prepare_grader(ROOT)
if not any((INSTALL_DEPENDENCIES, PREPARE_ENGINE, PREPARE_GRADER)):
    print('환경 준비를 실행하지 않았습니다.')

## 답안 생성

`RUN_GENERATION=True`로 설정합니다. 기존 `RUN_DIR`은 덮어쓰지 않으므로 다시 실행하려면 사용하지 않은 경로를 지정하세요. 전체 GPU 적재를 요청하며 RAM 80%, VRAM 97% 초과 시 경고 후 계속합니다. 모델·데이터 출처, 실제 실행 설정과 80 Expert 라우팅을 검증합니다.

In [31]:
if RUN_GENERATION:
    if not globals().get('_PUTARE_SETUP_READY', True):
        raise RuntimeError('환경 준비가 완료되지 않았습니다. 준비 셀을 다시 실행하세요.')
    if not MODEL.is_file():
        raise FileNotFoundError(f'모델 파일을 준비하세요: {MODEL}')
    from tools.native.gpu_memory import NvmlDevice
    device = NvmlDevice()
    try:
        major, minor = device.nvml.nvmlDeviceGetCudaComputeCapability(device.handle)
        gpu_info = {'name': device.name, 'uuid': device.uuid, 'compute_capability': f'{major}.{minor}'}
    finally:
        device.close()
    import json
    result = trial.generate(RUN_DIR, model=MODEL, engine=ENGINE)
    with (RUN_DIR / 'gpu-info.json').open('x', encoding='utf-8') as output:
        json.dump(gpu_info, output, ensure_ascii=False, indent=2)
    print(result)
else:
    print('생성 실행 대기 중입니다.')


1/3 math500-test-11 생성 시작
2/3 livecodebench:3674 생성 시작
3/3 livecodebench:3677 생성 시작
/content/Putare/0812c0c257e2-0cce743764ad/runs/gemma4-ordered-e80-trial3-colab-a100-001


## 채점

`RUN_GRADING=True`는 세 문제 생성과 모델 종료 후 실행합니다. LCB는 고정된 공식 채점기를 사용합니다. 생성된 코드는 별도 프로세스에서 Colab 사용자 권한으로 실행되며 보안 격리되지 않습니다. 수학은 최종 `\boxed{...}`의 분수를 정확히 비교합니다. 인식하지 못한 답은 수동 확인으로 남기며 공식 MATH-500 점수가 아닙니다.

In [32]:
if RUN_GRADING:
    if not globals().get('_PUTARE_SETUP_READY', True):
        raise RuntimeError('환경 준비가 완료되지 않았습니다.')
    print(trial.grade(RUN_DIR, model=MODEL, engine=ENGINE, venv=GRADER_VENV))
else:
    print('채점 대기 중입니다.')

{'complete': True, 'scope': 'MATH-500 1 + LCB 2', 'model_sha256': '760a302c4f67cb99dc28b6d195ae0ec6a623d8c37614fedb20df555b92180321', 'cases': [{'number': 1, 'id': 'math500-test-11', 'domain': 'math', 'status': 'unparsed', 'passed': None, 'answer': None, 'reference': '\\frac{3}{56}', 'method': 'exact rational comparison; not official MATH-500 grading', 'generated_tokens': 8793, 'stop_reason': 'eog', 'timings': {'decode_tokens_s': 97.96591192667489, 'load_s': 2.398296388, 'prefill_tokens_s': 143.85524002117694, 'ttft_s': 0.5427820000000001}, 'measurement': {'peak_process_tree_rss_bytes': 9876807680, 'session_peak_process_tree_rss_bytes': 9876807680, 'os_gpu_memory': {'peak_bytes': 18589155328, 'samples': 919, 'unavailable_reason': None, 'limitations': 'NVML 프로세스 메모리 표본 최대치; 물리적 최고치 보장 없음.', 'scope': 'session_to_date'}, 'memory_preflight': {'ram_total_bytes': 184644796416, 'ram_available_bytes': 152293941248, 'ram_used_bytes': 32350855168, 'ram_source': 'cgroup', 'ram_usage_fraction': 0.

## 결과 확인

생성 토큰 수, 실행 시간, decode 속도와 표본 메모리를 표시합니다. 수학 결과가 `unparsed`이면 원문을 직접 확인하세요. 생성 원문은 `RUN_SHOW_TEXT=True`일 때 표시합니다.

In [ ]:
import json
from IPython.display import Markdown, display
status_path = RUN_DIR / 'status.json'
summary_path = RUN_DIR / 'grading/summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else {}
by_id = {row['id']: row for row in summary.get('cases', [])}
if status_path.is_file():
    table = ['| 문제 | 결과 | 생성 토큰 | 시간(초) | decode tok/s | RAM GiB | GPU 표본 GiB | 수학 답 |',
             '|---|---|---:|---:|---:|---:|---:|---|']
    def fmt(value, scale=1):
        return '미측정' if value is None else f'{value / scale:.2f}'
    for number, ident in enumerate(trial.IDS, 1):
        path = RUN_DIR / 'answers' / f'{number:03d}.json'
        if not path.is_file():
            table.append(f'| {ident} | 생성 대기 | - | - | - | - | - | - |')
            continue
        answer = json.loads(path.read_text(encoding='utf-8'))
        row = by_id.get(ident, {})
        state = '통과' if row.get('passed') is True else '실패' if row.get('passed') is False else row.get('status', '채점 대기')
        measurement, timing = answer.get('measurement') or {}, answer.get('timings') or {}
        gpu = measurement.get('os_gpu_memory') or {}
        table.append(f"| {ident} | {state} | {len(answer.get('token_ids', []))} | {fmt(measurement.get('wall_s'))} | {fmt(timing.get('decode_tokens_s'))} | {fmt(measurement.get('peak_process_tree_rss_bytes'), 1024**3)} | {fmt(gpu.get('peak_bytes'), 1024**3)} | {row.get('answer') or '-'} |")
        if RUN_SHOW_TEXT:
            print(f'\n[{ident}]\n{answer.get("text", "")}')
    display(Markdown('\n'.join(table)))
else:
    print('아직 이 실행의 생성 결과가 없습니다.')